# STAB-A2 — Canonical candidate_v1 build

Run top-to-bottom. The full build is deliberately disabled until both contract suites and the smoke build succeed.

**Compute note:** STAB-A2 canonical union/dedup is CPU/I/O-bound; GPU acceleration is not expected to materially improve this stage. The GPU is logged but not used.

In [ ]:
# 1. Environment
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

def ensure_import(package, install_name=None):
    try:
        return __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', install_name or package])
        return __import__(package)

pl = ensure_import('polars')
pa = ensure_import('pyarrow')
psutil = ensure_import('psutil')
ensure_import('pytest')
print('Python:', platform.python_version())
print('Polars:', pl.__version__)
print('PyArrow:', pa.__version__)
print('CPU count:', os.cpu_count())
print('Host RAM GiB:', round(psutil.virtual_memory().total / 2**30, 2))
print('Available RAM GiB:', round(psutil.virtual_memory().available / 2**30, 2))
print('Disk free GiB:', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2))
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode == 0 else 'not available')
print('STAB-A2 canonical union/dedup is CPU/I/O-bound; GPU acceleration is not expected to materially improve this stage.')

In [ ]:
# 2. Repository setup — edit REPO if the checkout is elsewhere.
repo_candidates = [
    Path('/kaggle/working/Amazon_ML_Challenge_2026'),
    Path.cwd(),
]
REPO = next((p.resolve() for p in repo_candidates if (p / 'scripts' / '04e_build_canonical_candidates.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Repository checkout not found. Set REPO to the directory containing scripts/04e_build_canonical_candidates.py')
os.chdir(REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', REPO)
print('Git commit:', commit)
print('Git status:')
print(subprocess.check_output(['git', 'status', '--short'], text=True))

In [ ]:
# 3. A1 contract gate
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_candidates.py', '-q'], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    print('STAB-A1 CONTRACT TEST FAILURE')
    raise RuntimeError('STAB-A1 contract tests failed; candidate generation is blocked')

In [ ]:
# 4. A2 production-builder gate
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_candidate_builder.py', '-q'], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    print('STAB-A2 BUILDER TEST FAILURE')
    raise RuntimeError('STAB-A2 tests failed; candidate generation is blocked')

In [ ]:
# 5. Required artifact discovery — edit roots for the attached Kaggle datasets.
SPLIT = 'train'  # Run train and test as separate artifact builds.
CANDIDATE_ROOTS = [
    Path(os.environ.get('LOKESH_CANDIDATES_DIR', '/kaggle/input/lokesh-candidates/data/candidates')),
    Path(os.environ.get('LOKESH_CANDIDATES_V2_DIR', '/kaggle/input/lokesh-candidates-v2/data/candidates_v2')),
]
names = {
    'exact': f'{SPLIT}_exact_candidates.parquet',
    'dense': f'{SPLIT}_dense_candidates_K50.parquet',
    'word_tfidf': f'{SPLIT}_bm25_candidates_name_word_K50.parquet',
    'char_tfidf': f'{SPLIT}_char_candidates_name_char35_K50.parquet',
    'structured': f'{SPLIT}_structured_candidates.parquet',
}

def discover_required(name):
    matches = []
    for root in CANDIDATE_ROOTS:
        if root.exists():
            matches.extend(root.rglob(name))
    matches = sorted(set(path.resolve() for path in matches))
    if not matches:
        raise RuntimeError(f'MISSING REQUIRED RETRIEVER ARTIFACT:\n{name}\nroots={CANDIDATE_ROOTS}')
    if len(matches) > 1:
        raise RuntimeError(f'Ambiguous retriever artifact {name}: {matches}')
    return matches[0]

FILES = {retriever: discover_required(name) for retriever, name in names.items()}
for retriever, path in FILES.items():
    print(f'{retriever:15s} {path} bytes={path.stat().st_size:,}')

In [ ]:
# 6. Smoke build — samples queries, never independent candidate rows.
OUTPUT_ROOT = Path('/kaggle/working/artifacts/candidates/candidate_v1')
common = [
    sys.executable, 'scripts/04e_build_canonical_candidates.py',
    '--split', SPLIT,
    '--exact-path', str(FILES['exact']),
    '--dense-path', str(FILES['dense']),
    '--word-tfidf-path', str(FILES['word_tfidf']),
    '--char-tfidf-path', str(FILES['char_tfidf']),
    '--structured-path', str(FILES['structured']),
    '--output-dir', str(OUTPUT_ROOT),
    '--partitions', '64',
    '--batch-size', '1000000',
]
smoke = subprocess.run(common + ['--smoke', '--smoke-query-count', '100'], text=True)
if smoke.returncode != 0:
    raise RuntimeError('Smoke canonical build failed; full build is blocked')

In [ ]:
# 7. Smoke validation/report
smoke_manifest_path = OUTPUT_ROOT.parent / f'{OUTPUT_ROOT.name}_smoke' / SPLIT / 'manifest.json'
manifest = json.loads(smoke_manifest_path.read_text())
print('Manifest:', smoke_manifest_path)
print('Input rows by retriever:', manifest['input_rows_by_retriever'])
print('Selected/staged rows by retriever:', manifest['staged_rows_by_retriever'])
print('Canonical rows:', manifest['canonical_row_count'])
print('Duplicate rows collapsed:', manifest['duplicate_rows_collapsed'])
print('Completed partitions:', len(manifest['completed_partitions']))
print('Peak RSS bytes:', manifest['peak_rss_bytes'])
print('Builder runtime seconds:', manifest['builder_runtime_seconds'])
print('Current disk free GiB:', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2))

In [ ]:
# 8. Full build — intentionally opt in only after reviewing smoke output and disk projection.
RUN_FULL_BUILD = False
if not RUN_FULL_BUILD:
    print('Full build disabled. Review tests, smoke statistics, RAM, and disk; then set RUN_FULL_BUILD=True.')
else:
    full = subprocess.run(common, text=True)
    if full.returncode != 0:
        raise RuntimeError('Full canonical build failed')

In [ ]:
# 9. Full artifact summary (run only after a successful full build)
full_manifest_path = OUTPUT_ROOT / SPLIT / 'manifest.json'
if not full_manifest_path.exists():
    print('No full manifest yet:', full_manifest_path)
else:
    full_manifest = json.loads(full_manifest_path.read_text())
    print(json.dumps({
        'manifest': str(full_manifest_path),
        'status': full_manifest['status'],
        'canonical_rows': full_manifest['canonical_row_count'],
        'duplicates_collapsed': full_manifest['duplicate_rows_collapsed'],
        'completed_partitions': len(full_manifest['completed_partitions']),
        'config_fingerprint': full_manifest['config_fingerprint'],
        'dataset_fingerprint': full_manifest['dataset_fingerprint'],
    }, indent=2))